In [ ]:
!pip install vllm==0.10.0
!pip install triton==3.2.0
!pip install langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai
!pip uninstall -y openai
!pip install openai==1.90.0

In [ ]:
import os
GPT_API_KEY = user_secrets.get_secret("GPT_API_KEY")
BRAVE_API_KEY = user_secrets.get_secret("BRAVE_API_KEY")
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")

os.environ["WEB_SEARCH_SSL"] = str(False)
os.environ["GOOGLE_SEARCH_CX"] = "9501a956284f141ab"
os.environ["GOOGLE_SEARCH_API_KEY"] = GOOGLE_API_KEY
os.environ["BRAVE_SEARCH_API_KEY"] = BRAVE_API_KEY
os.environ["GPT_API_KEY"] = GPT_API_KEY

DOMAIN = "https://0d75e39e4aab.ngrok-free.app"
NGROK_PORT = 8002


In [ ]:
import requests
import io
import tarfile
import shutil
from typing import cast
def unpack(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) astar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_p(name: str):
    unpack(requests.get(f"{DOMAIN}/script/{name}").content, name)
def unpack_pl(*names: str):
    for name in names:
        unpack_p(name)
unpack_pl(
    "vllm_worker", "web_search", "server"
)

In [ ]:
EMBEDDING
_NAME = "intfloat/multilingual-e5-small"

In [8]:
def set_active(model_id: str):
    print(f"[Global] Switched to model {model_id}")
    if SEP in model_id:
        model_id = model_id.split(SEP)[0]
    for model in CLIENT_INFO["models"]:
        if model["id"].startswith(model_id):
            model["active"] = True
            model["scheduled"] = False
        else:
            model["active"] = False
            model["scheduled"] = False

In [10]:
class WebSearchWrapper:
    def __init__(self) -> None:
        self.web_search = Websearch(
            embedding_name=EMBEDDING_NAME, 
            device="cpu",
            chunk_size=1024, 
            chunk_overlap=128)
    async def start(self):
        await self.web_search.start()
    async def __call__(self, web_query: str, rag_query: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        k_pages = params.get("k_pages", 0)
        k_docs = params.get("k_docs", 0)
        domain_restrict = params.get("domain_restrict", False)
        print(f"[WS] k_pages: {k_pages} | k_docs {k_docs} | domain_restrict: {domain_restrict}")
        if k_pages == 0 or k_docs == 0:
            return [], []
        else:
            web_sources, rag_sources = await self.web_search(
                web_query=web_query, rag_query=rag_query,
                k_pages=k_pages, k_docs=k_docs,
                domain_restrict=domain_restrict, engine="brave",
                include_pdf=False, include_image=False
            )
            return web_sources, rag_sources

In [12]:
import threading
def thread_task():
    ws_pipeline = CustomQA()
    REQUEST_STORAGE: dict[str, tuple[str, KaggleRequest]] = {}
    async def pre_inference_function(request: KaggleRequest) -> ModelPreOutput:

        prompt, pre_output = await ws_pipeline.pre_inference(
            request["model_id"],
            request["text"],
            request["stream_id"],
            request["params"]
        )
        print(request["params"])
        REQUEST_STORAGE[request["stream_id"]] = (prompt, request)
        return pre_output
    
    async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
        prompt, request = REQUEST_STORAGE.pop(stream_id)
        generator = await ws_pipeline.inference(prompt, request)
        return generator
    
    app = construct_app(
        server_domain=DOMAIN,
        info=CLIENT_INFO,
        pre_inference=pre_inference_function,
        inferece=inference_function,
        init_tasks=[
            ws_pipeline.start(), 
            ws_pipeline.preload(PRELOAD_MODEL)
        ]
    )
    
    uvicorn.run(app, port=NGROK_PORT)
thread = threading.Thread(target=thread_task, daemon=True)
thread.start()

2025-08-27 07:19:53.067964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756279193.098868    1828 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756279193.108357    1828 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO:     Started server process [1786]
INFO:     Waiting for application startup.


Domain: https://ce3c37786af2.ngrok-free.app
[VectorCache] Initialized with domain: https://0d75e39e4aab.ngrok-free.app
[CustomQA] Vector cache client initialized with domain: https://0d75e39e4aab.ngrok-free.app
[Global] Switched to model Qwen/Qwen3-4B
[VLLM Engine] Server is starting: Cannot connect to host 127.0.0.1:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
[VLLM Engine] Server is starting: Cannot connect to host 127.0.0.1:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]


2025-08-27 07:20:08.642942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756279208.668471    1844 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756279208.676526    1844 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[VLLM Engine] Server is starting: Cannot connect to host 127.0.0.1:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
[VLLM Engine] Server is starting: Cannot connect to host 127.0.0.1:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
INFO 08-27 07:20:13 [__init__.py:235] Automatically detected platform cuda.
INFO 08-27 07:20:14 [server_setup.py:22] [vLLM] vLLM API server version 0.10.0
INFO 08-27 07:20:14 [server_setup.py:23] [vLLM] Server started at 1844
INFO 08-27 07:20:14 [server_setup.py:24] Available route are:
INFO 08-27 07:20:14 [server_setup.py:30] Route: /openapi.json, Methods: GET, HEAD
INFO 08-27 07:20:14 [server_setup.py:30] Route: /docs, Methods: GET, HEAD
INFO 08-27 07:20:14 [server_setup.py:30] Route: /docs/oauth2-redirect, Methods: GET, HEAD
INFO 08-27 07:20:14 [server_setup.py:30] Route: /redoc, Methods: GET, HEAD
INFO 08-27 07:20:14 [server_setup.py:30] Route: /health, Methods: GET
INFO 08-27 07:20:14 [server_setup.py:30] Route: /init, Methods: POST


INFO:     Started server process [1844]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


...

(VllmWorkerProcess pid=1871) INFO 08-27 07:21:25 [custom_all_reduce.py:196] Registering 1387 cuda graph addresses


Capturing CUDA graph shapes: 100%|██████████| 19/19 [00:22<00:00,  1.18s/it]


INFO 08-27 07:21:25 [custom_all_reduce.py:196] Registering 1387 cuda graph addresses
(VllmWorkerProcess pid=1871) INFO 08-27 07:21:26 [model_runner.py:1537] Graph capturing finished in 23 secs, took 0.24 GiB
INFO 08-27 07:21:26 [model_runner.py:1537] Graph capturing finished in 23 secs, took 0.24 GiB
INFO 08-27 07:21:26 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 32.74 seconds
INFO:     127.0.0.1:57808 - "POST /init HTTP/1.1" 200 OK
[VLLM Controller] Server started 200: Sucess
INFO 08-27 07:21:27 [chat_utils.py:473] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO:     127.0.0.1:35544 - "POST /generate HTTP/1.1" 200 OK
INFO 08-27 07:21:27 [async_llm_engine.py:209] Added request cfdcfbeb7b5045128fe8f11a35eeb452.
INFO 08-27 07:21:28 [async_llm_engine.py:177] Finished request cfdcfbeb7b5045128fe8f11a35eeb452.


INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


[QueryRewrite] Search strategy: {'type_search': 'vector_db', 'key_word': [{'school_id': 'UET', 'section': 'thong_tin_chung'}]}
[QueryRewrite] Searching vector DB: UET/thong_tin_chung
[QueryRewrite] Found 1 documents from vector DB
[QueryRewrite] Final result: 1 vector sources, web_query: 'địa chỉ uet'
[CustomQA] Using vector DB sources: 1 documents
{'domain_restrict': False, 'k_docs': 5, 'k_pages': 3, 'max_tokens': 2048, 'temperature': 0.5, 'top_p': 0.9}
INFO:     2405:4802:1bd5:1b30:11a6:2f7d:9bf0:3371:0 - "POST /pre_inference HTTP/1.1" 200 OK
INFO:     2405:4802:1bd5:1b30:11a6:2f7d:9bf0:3371:0 - "POST /inference/c4065d5b-5f32-4abd-858b-eb7c561973ff HTTP/1.1" 200 OK
INFO:     127.0.0.1:57920 - "POST /generate HTTP/1.1" 200 OK
INFO 08-27 07:22:04 [async_llm_engine.py:209] Added request 8dbc8d30a8124fe6807cab98225790b7.
INFO 08-27 07:22:04 [metrics.py:386] Avg prompt throughput: 7.7 tokens/s, Avg generation throughput: 0.3 tokens/s, Running: 1 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU